Chiến Lược ATR + Moving Average
Cách hoạt động:
Fit: Kết hợp ATR với MA để lọc tín hiệu

=> OG
Mua: MA Fast > MA Slow VÀ ATR < threshold (thị trường ít biến động)
Bán: MA Fast < MA Slow VÀ ATR > threshold (thị trường biến động mạnh)

In [1]:
import sys
sys.path.append("../Common")

import CommonBinance

In [2]:
symbol = 'ETHUSDT'
from_date = '2025-06-01'
to_date = '2025-08-28'
interval = '1d'
data = CommonBinance.CommonBinance.loaddataBinance_FromTo(symbol, from_date, to_date, interval)
# 2025-08-28	4506.71	4633.97	4467.63	4552.40	3.016636e+05


In [3]:
# Import các thư viện cần thiết
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

In [4]:
import numpy as np
from itertools import combinations
import talib

# Đảm bảo cột 'Datetime' được hiểu là kiểu dữ liệu datetime và đã sắp xếp
data['Datetime'] = pd.to_datetime(data['Datetime'])
data.sort_values(by='Datetime', inplace=True)

# Thêm MA và ATR vào DataFrame
data['MA'] = talib.SMA(data['Close'], timeperiod=20)
data['ATR'] = talib.ATR(data['High'], data['Low'], data['Close'], timeperiod=14)

# Loại bỏ NaN
# data.dropna(inplace=True)

data.fillna(data.mean(), inplace=True)

# Khởi tạo biến mục tiêu 'y' là giá đóng cửa
y = data['Close']

# Định nghĩa cơ bản của các đặc trưng
# 'Open', 'High', 'Low', 'Volume', 'Momentum', 'RSI'
basic_features = ['Open', 'High', 'Low', 'Volume', 'MA', 'ATR']

# Tạo tất cả tổ hợp có thể từ basic_features
# Sử dụng itertools để tạo tất cả các tổ hợp của các đặc trưng
features_combinations = []
for r in range(1, len(basic_features) + 1):
    for combo in combinations(basic_features, r):
        features_combinations.append(list(combo))

# Luu trữ kết quả MSE cho từng tổ hợp
mse_scores = {}

# Kiểm tra từng kết hợp đặc trưng
for features in features_combinations:
    X = data[features].copy() # Copy dữ liệu để tránh thay đổi gốc
	# Thay thế NaN bằng giá trị trung bình của cột
    X.fillna(X.mean(), inplace=True)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = DecisionTreeRegressor(random_state=42)
    model.fit(X_train, y_train)
    
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    
    mse_scores[str(features)] = mse

print(mse_scores)
# Tìm kết hợp đặc trưng với MSE thấp nhất
best_features = min(mse_scores, key=mse_scores.get)
print(f"Best features: {best_features} with MSE: {mse_scores[best_features]}")

{"['Open']": 25194.92097777776, "['High']": 6116.9097499999925, "['Low']": 31005.40113333329, "['Volume']": 1145140.7566499999, "['MA']": 31635.333759297042, "['ATR']": 103617.15421666666, "['Open', 'High']": 9889.274299999997, "['Open', 'Low']": 34468.74718888883, "['Open', 'Volume']": 25289.141461111096, "['Open', 'MA']": 30119.161811111087, "['Open', 'ATR']": 30274.797672222227, "['High', 'Low']": 8089.940588888866, "['High', 'Volume']": 5460.077983333325, "['High', 'MA']": 10275.690355555551, "['High', 'ATR']": 12381.863372222215, "['Low', 'Volume']": 18452.87209444445, "['Low', 'MA']": 32708.471838888836, "['Low', 'ATR']": 31340.96317222218, "['Volume', 'MA']": 55085.813, "['Volume', 'ATR']": 116055.53787222227, "['MA', 'ATR']": 30777.340683333325, "['Open', 'High', 'Low']": 10956.196772222209, "['Open', 'High', 'Volume']": 12336.51462777776, "['Open', 'High', 'MA']": 13018.755533333322, "['Open', 'High', 'ATR']": 15802.64188333331, "['Open', 'Low', 'Volume']": 21239.21929444444, 